In [ ]:
# Activacion de entorno

source mi_entorno/bin/activate

# Abrir Jupyter

jupyter notebook

In [9]:
# ================================
# 1. SETUP
# ================================

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import pandas as pd
import json
import time

options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

# Página base (clave para evitar bloqueo)
driver.get("https://resultadoelectoral.onpe.gob.pe/main/presidenciales")
time.sleep(5)

# ================================
# 2. REGIONES
# ================================

departamentos = [
    ("010000","AMAZONAS"),
    ("020000","ANCASH"),
    ("030000","APURIMAC"),
    ("040000","AREQUIPA"),
    ("050000","AYACUCHO"),
    ("060000","CAJAMARCA"),
    ("240000","CALLAO"),
    ("070000","CUSCO"),
    ("080000","HUANCAVELICA"),
    ("090000","HUANUCO"),
    ("100000","ICA"),
    ("110000","JUNIN"),
    ("120000","LA LIBERTAD"),
    ("130000","LAMBAYEQUE"),
    ("140000","LIMA"),
    ("150000","LORETO"),
    ("160000","MADRE DE DIOS"),
    ("170000","MOQUEGUA"),
    ("180000","PASCO"),
    ("190000","PIURA"),
    ("200000","PUNO"),
    ("210000","SAN MARTIN"),
    ("220000","TACNA"),
    ("230000","TUMBES"),
    ("250000","UCAYALI")
]

# ================================
# 3. EXTRAER PROVINCIAS
# ================================

provincias = []

for ubigeo_dep, nombre_dep in departamentos:
    
    url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/ubigeos/provincias?idEleccion=10&idAmbitoGeografico=1&idUbigeoDepartamento={ubigeo_dep}"
    
    print(f"Provincias {nombre_dep}...")

    driver.get(url)
    time.sleep(2)

    texto = driver.find_element("tag name", "body").text

    try:
        data = json.loads(texto)["data"]

        for p in data:
            provincias.append({
                "departamento": nombre_dep,
                "ubigeo_dep": ubigeo_dep,
                "ubigeo_prov": p["ubigeo"],
                "provincia": p["nombre"]
            })

    except Exception as e:
        print(f"❌ Error provincias {nombre_dep}: {e}")

df_provincias = pd.DataFrame(provincias)

print("✅ Provincias cargadas")

# ================================
# 4. VOTOS POR PROVINCIA
# ================================

votos = []

for i, row in df_provincias.iterrows():

    url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/eleccion-presidencial/participantes-ubicacion-geografica-nombre?tipoFiltro=ubigeo_nivel_02&idAmbitoGeografico=1&ubigeoNivel1={row['ubigeo_dep']}&ubigeoNivel2={row['ubigeo_prov']}&idEleccion=10"

    print(f"Votos {row['provincia']} ({row['departamento']})")

    driver.get(url)
    time.sleep(2)

    texto = driver.find_element("tag name", "body").text

    try:
        data = json.loads(texto)["data"]

        for fila in data:
            votos.append({
                "departamento": row["departamento"],
                "provincia": row["provincia"],
                "candidato": fila["nombreCandidato"],
                "partido": fila["nombreAgrupacionPolitica"],
                "votos": fila.get("totalVotosValidos", 0)
            })

    except Exception as e:
        print(f"❌ Error votos {row['provincia']}: {e}")

df_votos = pd.DataFrame(votos)

print("✅ Votos cargados")

# ================================
# 5. ACTAS POR PROVINCIA
# ================================

actas = []

for i, row in df_provincias.iterrows():

    url = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/resumen-general/totales?idAmbitoGeografico=1&idEleccion=10&tipoFiltro=ubigeo_nivel_02&idUbigeoDepartamento={row['ubigeo_dep']}&idUbigeoProvincia={row['ubigeo_prov']}"

    print(f"Actas {row['provincia']} ({row['departamento']})")

    driver.get(url)
    time.sleep(2)

    texto = driver.find_element("tag name", "body").text

    try:
        data = json.loads(texto)["data"]

        actas.append({
            "departamento": row["departamento"],
            "provincia": row["provincia"],
            "actas_total": data["totalActas"],
            "actas_procesadas": data["contabilizadas"],
            "avance_pct": data["actasContabilizadas"],
            "votos_validos_prov": data["totalVotosValidos"]
        })

    except Exception as e:
        print(f"❌ Error actas {row['provincia']}: {e}")

df_actas = pd.DataFrame(actas)

print("✅ Actas cargadas")

# ================================
# 6. MERGE FINAL
# ================================

df_final = df_votos.merge(
    df_actas,
    on=["departamento", "provincia"]
)

# ================================
# 7. EXPORTAR
# ================================

df_final.to_csv("resultado_provincial_completo.csv", index=False)

print("\n🔥 LISTO")
print("Archivo generado: resultado_provincial_completo.csv")

# ================================
# 8. PREVIEW
# ================================

df_final.head()

Provincias AMAZONAS...
Provincias ANCASH...
Provincias APURIMAC...
Provincias AREQUIPA...
Provincias AYACUCHO...
Provincias CAJAMARCA...
Provincias CALLAO...
Provincias CUSCO...
Provincias HUANCAVELICA...
Provincias HUANUCO...
Provincias ICA...
Provincias JUNIN...
Provincias LA LIBERTAD...
Provincias LAMBAYEQUE...
Provincias LIMA...
Provincias LORETO...
Provincias MADRE DE DIOS...
Provincias MOQUEGUA...
Provincias PASCO...
Provincias PIURA...
Provincias PUNO...
Provincias SAN MARTIN...
Provincias TACNA...
Provincias TUMBES...
Provincias UCAYALI...
✅ Provincias cargadas
Votos BAGUA (AMAZONAS)
Votos BONGARÁ (AMAZONAS)
Votos CHACHAPOYAS (AMAZONAS)
Votos CONDORCANQUI (AMAZONAS)
Votos LUYA (AMAZONAS)
Votos RODRÍGUEZ DE MENDOZA (AMAZONAS)
Votos UTCUBAMBA (AMAZONAS)
Votos AIJA (ANCASH)
Votos ANTONIO RAIMONDI (ANCASH)
Votos ASUNCIÓN (ANCASH)
Votos BOLOGNESI (ANCASH)
Votos CARHUAZ (ANCASH)
Votos CARLOS FERMÍN FITZCARRALD (ANCASH)
Votos CASMA (ANCASH)
Votos CORONGO (ANCASH)
Votos HUARAZ (ANCASH)

,departamento,provincia,candidato,partido,votos,actas_total,actas_procesadas,avance_pct,votos_validos_prov
0,AMAZONAS,BAGUA,ROBERTO HELBERT SANCHEZ PALOMINO,JUNTOS POR EL PERÚ,13037,250,231,92.4,30259
1,AMAZONAS,BAGUA,,VOTOS EN BLANCO,6312,250,231,92.4,30259
2,AMAZONAS,BAGUA,KEIKO SOFIA FUJIMORI HIGUCHI,FUERZA POPULAR,4629,250,231,92.4,30259
3,AMAZONAS,BAGUA,RICARDO PABLO BELMONT CASSINELLI,PARTIDO CÍVICO OBRAS,2651,250,231,92.4,30259
4,AMAZONAS,BAGUA,,VOTOS NULOS,2523,250,231,92.4,30259


In [10]:
# ================================
# 1. LIMPIEZA BÁSICA
# ================================

import numpy as np

df = df_final.copy()

# Evitar divisiones por 0
df["avance_pct"] = df["avance_pct"].replace(0, np.nan)

# ================================
# 2. EXTRAPOLACIÓN PROVINCIAL
# ================================

df["factor_expansion"] = 100 / df["avance_pct"]

df["votos_estimados"] = df["votos"] * df["factor_expansion"]

# ================================
# 3. VOTOS RESTANTES (insight clave)
# ================================

df["votos_restantes"] = df["votos_estimados"] - df["votos"]

# ================================
# 4. AGREGAR A NIVEL NACIONAL
# ================================

df_nacional = (
    df
    .groupby(["candidato", "partido"], as_index=False)
    .agg({
        "votos": "sum",
        "votos_estimados": "sum",
        "votos_restantes": "sum"
    })
)

# ================================
# 5. IDENTIFICAR NO VÁLIDOS
# ================================

no_validos = ["VOTOS EN BLANCO", "VOTOS NULOS"]

# Total votos válidos estimados
total_validos = df_nacional[
    ~df_nacional["partido"].isin(no_validos)
]["votos_estimados"].sum()

# ================================
# 6. % VOTOS VÁLIDOS
# ================================

df_nacional["pct_votos_validos"] = df_nacional.apply(
    lambda row: row["votos_estimados"] / total_validos * 100
    if row["partido"] not in no_validos else None,
    axis=1
)

# ================================
# 7. ORDENAR
# ================================

df_nacional = df_nacional.sort_values(
    by="votos_estimados",
    ascending=False
)

# ================================
# 8. TOP 20
# ================================

top20 = df_nacional.head(20)

print("\n🔥 TOP 20 NACIONAL (EXTRAPOLACIÓN PROVINCIAL)\n")
print(top20)

# ================================
# 9. EXPORTAR
# ================================

top20.to_csv("top20_nacional_provincial.csv", index=False)

print("\n✅ Archivo generado: top20_nacional_provincial.csv")


🔥 TOP 20 NACIONAL (EXTRAPOLACIÓN PROVINCIAL)

                               candidato  \
18          KEIKO SOFIA FUJIMORI HIGUCHI   
0                                          
31      ROBERTO HELBERT SANCHEZ PALOMINO   
27  RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA   
15                JORGE NIETO MONTESINOS   
29      RICARDO PABLO BELMONT CASSINELLI   
8          CARLOS GONSALO ALVAREZ LOAYZA   
24         PABLO ALFONSO LOPEZ CHAU NAVA   
1                                          
20             MARIA SOLEDAD PEREZ TELLO   
3    ALFONSO CARLOS ESPA Y GARCES-ALVEAR   
19            LUIS FERNANDO OLIVERA VEGA   
17                 JOSE LEON LUNA GALVEZ   
37                 YONHY LESCANO ANCIETA   
9                    CESAR ACUÑA PERALTA   
26        PITTER ENRIQUE VALDERRAMA PEÑA   
13         GEORGE PATRICK FORSYTH SOMMER   
21        MARIO ENRIQUE VIZCARRA CORNEJO   
14              HERBERT CALLER GUTIERREZ   
32       RONALD DARWIN ATENCIO SOTOMAYOR   

                            

In [12]:
# ================================
# 15. EXTRANJERO (SETUP)
# ================================

extranjero = [
    ("910000","AFRICA"),
    ("920000","AMERICA"),
    ("930000","ASIA"),
    ("940000","EUROPA"),
    ("950000","OCEANIA")
]

votos_ext = []
actas_ext = []

# ================================
# 16. LOOP EXTRANJERO
# ================================

for ubigeo, nombre in extranjero:

    print(f"Procesando EXTERIOR {nombre}...")

    # ------------------------
    # VOTOS
    # ------------------------
    url_votos = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/eleccion-presidencial/participantes-ubicacion-geografica-nombre?tipoFiltro=ubigeo_nivel_01&idAmbitoGeografico=2&ubigeoNivel1={ubigeo}&idEleccion=10"

    driver.get(url_votos)
    time.sleep(3)

    texto_votos = driver.find_element("tag name", "body").text

    try:
        data_votos = json.loads(texto_votos)["data"]

        for fila in data_votos:
            votos_ext.append({
                "departamento": f"EXTERIOR_{nombre}",
                "provincia": f"EXTERIOR_{nombre}",
                "candidato": fila["nombreCandidato"],
                "partido": fila["nombreAgrupacionPolitica"],
                "votos": fila.get("totalVotosValidos", 0)
            })

    except Exception as e:
        print(f"❌ Error votos EXTERIOR {nombre}: {e}")
        continue

    # ------------------------
    # ACTAS
    # ------------------------
    url_actas = f"https://resultadoelectoral.onpe.gob.pe/presentacion-backend/resumen-general/totales?idAmbitoGeografico=2&idEleccion=10&tipoFiltro=ubigeo_nivel_01&idUbigeoDepartamento={ubigeo}"

    driver.get(url_actas)
    time.sleep(3)

    texto_actas = driver.find_element("tag name", "body").text

    try:
        data_actas = json.loads(texto_actas)["data"]

        actas_ext.append({
            "departamento": f"EXTERIOR_{nombre}",
            "provincia": f"EXTERIOR_{nombre}",
            "actas_total": data_actas["totalActas"],
            "actas_procesadas": data_actas["contabilizadas"],
            "avance_pct": data_actas["actasContabilizadas"],  # ✅ correcto
            "votos_validos_prov": data_actas["totalVotosValidos"]
        })

    except Exception as e:
        print(f"❌ Error actas EXTERIOR {nombre}: {e}")

# ================================
# 17. DATAFRAME EXTRANJERO
# ================================

df_votos_ext = pd.DataFrame(votos_ext)
df_actas_ext = pd.DataFrame(actas_ext)

df_ext = df_votos_ext.merge(df_actas_ext, on=["departamento","provincia"])

# ================================
# 18. UNIR PERÚ + EXTRANJERO (SIN DUPLICAR)
# ================================

df_total = pd.concat([df_final, df_ext], ignore_index=True)

# ================================
# 18.1 BASE INMUTABLE
# ================================

df_base = df_total.copy()

df_base["avance_pct"] = df_base["avance_pct"].replace(0, np.nan)

# 🔥 AQUÍ ESTÁ LA EXTRAPOLACIÓN
df_base["votos_estimados"] = df_base["votos"] / (df_base["avance_pct"] / 100)

df_base["votos_restantes"] = df_base["votos_estimados"] - df_base["votos"]

df_base.to_csv("base_intermedia_proyeccion.csv", index=False)

print("✅ Base intermedia guardada")

# ================================
# 18.2 TOTAL POR REGIÓN (para shares)
# ================================

totales_region = (
    df_base
    .groupby("departamento")["votos_estimados"]
    .sum()
    .reset_index()
    .rename(columns={"votos_estimados": "total_region"})
)

df_base = df_base.merge(totales_region, on="departamento")

# ================================
# 18.3 MÉTRICAS PROVINCIALES
# ================================

df_base["pct_region_actual"] = (
    df_base["votos"] / df_base["total_region"] * 100
)

df_base["pct_region_proyectado"] = (
    df_base["votos_estimados"] / df_base["total_region"] * 100
)

# ================================
# 18.4 FILTRO OPCIONAL (TOP CANDIDATOS)
# ================================

focus = [
    "RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA",
    "ROBERTO HELBERT SANCHEZ PALOMINO",
    "JORGE NIETO MONTESINOS"
]

df_resultado = df_base[df_base["candidato"].isin(focus)].copy()

# ================================
# 18.5 EXPORT FINAL (PROVINCIAL)
# ================================

df_resultado = df_resultado[[
    "candidato",
    "departamento",
    "provincia",
    "votos",
    "votos_estimados",
    "votos_restantes",
    "avance_pct",
    "total_region",
    "pct_region_actual",
    "pct_region_proyectado"
]]

df_resultado.to_csv("proyeccion_provincial_completa.csv", index=False)

print("\n✅ Archivo generado:")
print("proyeccion_provincial_completa.csv")

# ================================
# 19. AGREGADO NACIONAL (OPCIONAL)
# ================================

df_nacional = (
    df_base
    .groupby(["candidato", "partido"], as_index=False)
    .agg({"votos_estimados": "sum"})
)

df_nacional = df_nacional.sort_values(by="votos_estimados", ascending=False)

df_nacional.to_csv("nacional_resumen.csv", index=False)

print("✅ nacional_resumen.csv generado")

Procesando EXTERIOR AFRICA...
Procesando EXTERIOR AMERICA...
Procesando EXTERIOR ASIA...
Procesando EXTERIOR EUROPA...
Procesando EXTERIOR OCEANIA...
✅ Base intermedia guardada

✅ Archivo generado:
proyeccion_provincial_completa.csv
✅ nacional_resumen.csv generado


In [1]:
df_sanchez = df[df["candidato"] == "ROBERTO HELBERT SANCHEZ PALOMINO"]

df_sanchez.sort_values("votos_restantes", ascending=False).head(20)

NameError: name 'df' is not defined

In [4]:
df["avance_pct"].describe()

count    7296.000000
mean       69.131328
std        28.645066
min         0.734000
25%        55.653000
50%        80.358500
75%        92.628000
max       100.000000
Name: avance_pct, dtype: float64

In [5]:
df[df["avance_pct"] < 30].groupby("candidato")["votos"].sum().sort_values(ascending=False)

candidato
ROBERTO HELBERT SANCHEZ PALOMINO            33613
                                            31721
KEIKO SOFIA FUJIMORI HIGUCHI                16475
RICARDO PABLO BELMONT CASSINELLI            11481
PABLO ALFONSO LOPEZ CHAU NAVA                5403
CARLOS GONSALO ALVAREZ LOAYZA                4326
JORGE NIETO MONTESINOS                       4017
RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA         3536
JOSE LEON LUNA GALVEZ                        2664
RONALD DARWIN ATENCIO SOTOMAYOR              2236
ALFONSO CARLOS ESPA Y GARCES-ALVEAR          1746
MARIA SOLEDAD PEREZ TELLO                    1485
LUIS FERNANDO OLIVERA VEGA                   1472
YONHY LESCANO ANCIETA                        1372
CESAR ACUÑA PERALTA                          1298
GEORGE PATRICK FORSYTH SOMMER                1283
HERBERT CALLER GUTIERREZ                     1088
MARIO ENRIQUE VIZCARRA CORNEJO               1048
CHARLIE CARRASCO SALAZAR                      992
VLADIMIR ROY CERRON ROJAS               

In [6]:
df_bajo = df[df["avance_pct"] < 30]

df_bajo.groupby("candidato")["votos"].sum().sort_values(ascending=False).head(5)

candidato
ROBERTO HELBERT SANCHEZ PALOMINO    33613
                                    31721
KEIKO SOFIA FUJIMORI HIGUCHI        16475
RICARDO PABLO BELMONT CASSINELLI    11481
PABLO ALFONSO LOPEZ CHAU NAVA        5403
Name: votos, dtype: int64

In [7]:
df_alto = df[df["avance_pct"] > 80]

df_alto.groupby("candidato")["votos"].sum().sort_values(ascending=False).head(5)

candidato
                                        1845903
KEIKO SOFIA FUJIMORI HIGUCHI            1789646
RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA    1565954
JORGE NIETO MONTESINOS                  1438624
RICARDO PABLO BELMONT CASSINELLI        1119108
Name: votos, dtype: int64

In [8]:
import numpy as np
import pandas as pd
import time

# ================================
# 1. BASE LIMPIA (INPUT)
# ================================

df_mc_base = df_final.copy()

# evitar división por cero
df_mc_base["avance_pct"] = df_mc_base["avance_pct"].replace(0, np.nan)

# extrapolación determinística base
df_mc_base["votos_estimados"] = (
    df_mc_base["votos"] / (df_mc_base["avance_pct"] / 100)
)

# residual (lo que falta por distribuir)
df_mc_base["votos_restantes"] = (
    df_mc_base["votos_estimados"] - df_mc_base["votos"]
)

# limpieza de seguridad
df_mc_base = df_mc_base.dropna(subset=["votos_restantes"])
df_mc_base["votos_restantes"] = df_mc_base["votos_restantes"].clip(lower=0)

# ================================
# 2. PARÁMETROS MONTE CARLO
# ================================

N = 5000
step_print = 500
start_time = time.time()

resultados = []

# nivel de agregación territorial (AJUSTA AQUÍ)
group_cols = ["provincia"]   # o ["departamento"] o ["distrito"]

# nombres exactos candidatos
RLA = "RAFAEL BERNARDO LÓPEZ ALIAGA CAZORLA"
SANCH = "ROBERTO HELBERT SANCHEZ PALOMINO"
NIETO = "JORGE NIETO MONTESINOS"

# ================================
# 3. MONTE CARLO
# ================================

for i in range(N):

    sim = df_mc_base.copy()

    # ----------------------------
    # shock aleatorio territorial
    # ----------------------------
    ruido = np.random.lognormal(mean=0, sigma=0.12, size=len(sim))
    sim["peso_sim"] = ruido

    # normalizar dentro de grupo territorial
    sim["peso_sim"] = sim.groupby(group_cols)["peso_sim"].transform(
        lambda x: x / x.sum()
    )

    # ----------------------------
    # redistribución de votos restantes
    # ----------------------------
    sim["votos_simulados"] = (
        sim["votos"] + sim["peso_sim"] * sim["votos_restantes"]
    )

    # ----------------------------
    # agregación nacional
    # ----------------------------
    nacional = sim.groupby("candidato")["votos_simulados"].sum()

    votos_rla = nacional.get(RLA, 0)
    votos_sanchez = nacional.get(SANCH, 0)
    votos_nieto = nacional.get(NIETO, 0)

    resultados.append({
        "rla": votos_rla,
        "sanchez": votos_sanchez,
        "nieto": votos_nieto,
        "rla_gana": votos_rla > votos_sanchez
    })

    # ----------------------------
    # progreso
    # ----------------------------
    if (i + 1) % step_print == 0:
        elapsed = time.time() - start_time
        eta = (elapsed / (i + 1)) * (N - i - 1)

        print(
            f"🔄 {i+1}/{N} | "
            f"{(i+1)/N:.1%} | "
            f"ETA: {eta:.1f}s"
        )

# ================================
# 4. RESULTADOS FINALES
# ================================

df_res = pd.DataFrame(resultados)

df_res["diff"] = df_res["rla"] - df_res["sanchez"]

print("\n🔥 RESULTADOS MONTE CARLO\n")

print("Probabilidad RLA gana:",
      round(df_res["rla_gana"].mean() * 100, 2), "%")

print("Probabilidad Sánchez gana:",
      round((1 - df_res["rla_gana"].mean()) * 100, 2), "%")

print("\n📊 Distribución diferencia (RLA - Sánchez):")

print(df_res["diff"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]))

🔄 500/5000 | 10.0% | ETA: 120.5s
🔄 1000/5000 | 20.0% | ETA: 108.0s
🔄 1500/5000 | 30.0% | ETA: 95.8s
🔄 2000/5000 | 40.0% | ETA: 83.5s
🔄 2500/5000 | 50.0% | ETA: 69.2s
🔄 3000/5000 | 60.0% | ETA: 55.7s
🔄 3500/5000 | 70.0% | ETA: 41.7s
🔄 4000/5000 | 80.0% | ETA: 27.8s
🔄 4500/5000 | 90.0% | ETA: 13.9s
🔄 5000/5000 | 100.0% | ETA: 0.0s

🔥 RESULTADOS MONTE CARLO

Probabilidad RLA gana: 0.0 %
Probabilidad Sánchez gana: 100.0 %

📊 Distribución diferencia (RLA - Sánchez):
0.05   -61386.158209
0.25   -61145.889474
0.50   -60969.891891
0.75   -60781.161285
0.95   -60492.315825
Name: diff, dtype: float64


In [12]:
import pandas as pd

df_res = pd.DataFrame(resultados)

# Probabilidades
prob_rla = df_res["gana_rla"].mean()
prob_sanchez = 1 - prob_rla

# Diferencia promedio
df_res["diff"] = df_res["rla"] - df_res["sanchez"]
diff_media = df_res["diff"].mean()

print("\n🔥 RESULTADOS MONTE CARLO\n")
print(f"Probabilidad RLA gana: {prob_rla:.2%}")
print(f"Probabilidad Sánchez gana: {prob_sanchez:.2%}")
print(f"Diferencia promedio (RLA - Sánchez): {diff_media:,.0f} votos")

# Intervalos
print("\n📊 Distribución de diferencia (RLA - Sánchez):")
print(df_res["diff"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]))


🔥 RESULTADOS MONTE CARLO

Probabilidad RLA gana: 100.00%
Probabilidad Sánchez gana: 0.00%
Diferencia promedio (RLA - Sánchez): 178,549 votos

📊 Distribución de diferencia (RLA - Sánchez):
0.05    172890.242109
0.25    176312.255743
0.50    178474.430879
0.75    180806.148868
0.95    184019.540674
Name: diff, dtype: float64
